# RevX Call Intelligence — Analysis Notebook (Shared / Anonymized)

Reproduces the conversion-likelihood (XGBoost) and call-value (regression) models on the Ringba ICP enrichment data, plus Google Ads performance. Edit freely.

**Privacy note:** account/campaign names are removed. `campaign_prefix` is replaced with generic IDs (`AC01`–`AC12`), and Google Ads campaigns are consolidated to campaign-type totals. No account or campaign is identifiable.

**Data files** (download from the repo `data/` folder, then upload here): `icp_calls.csv`, `google_demographics.csv`, `google_campaigns.csv`, `google_geo.csv`, `google_hour.csv`.

## 1 · Setup & load data
Run this cell, then pick `icp_calls.csv` (and optionally the google_*.csv files) in the upload dialog.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    from google.colab import files
    up = files.upload()  # choose icp_calls.csv (+ google_*.csv)
except Exception:
    pass
df = pd.read_csv('icp_calls.csv')
print(df.shape)
df.head()

## 2 · Quick EDA
Conversion rate, enrichment coverage, revenue.

In [ ]:
print('calls:', len(df))
print('conversion rate: %.1f%%' % (100*df.converted.mean()))
print('enriched: %.1f%%' % (100*df.enriched.mean()))
print('mean revenue/call: $%.2f' % df.revenue.mean())
print('mean revenue | converted: $%.2f' % df.loc[df.converted==1,'revenue'].mean())
df[['age','income_rank','networth_rank','home_value_usd','vehicle_count']].describe()

### Conversion rate by segment (example: income band)

In [ ]:
seg = df.dropna(subset=['income_rank']).copy()
seg['band'] = pd.cut(seg['income_rank'], [0,3,6,9,11,13], labels=['Lowest','Low-mid','Mid-high','High','Highest'])
g = seg.groupby('band', observed=True)['converted'].agg(['mean','size'])
g['mean'] *= 100
ax = g['mean'].plot(kind='bar', title='Conversion % by income band', ylabel='conversion %')
plt.axhline(100*df.converted.mean(), ls='--', c='b'); plt.tight_layout(); plt.show()
g

## 3 · Feature matrix
Numeric + one-hot categoricals. Edit the lists to experiment. (`campaign_prefix` here is an anonymized account ID, not a real campaign name.)

In [ ]:
NUM = ['age','income_rank','networth_rank','home_value_usd','education_rank','num_children_rank',
       'length_of_residence','zip_pct_white','zip_pct_black','zip_pct_hispanic','zip_pct_asian',
       'investment_interest','vehicle_count','newest_vehicle_year','mortgage_amount','mortgage_ltv',
       'property_market_value','year_built','call_length_s','hour','dow','enriched']
CAT = ['state','campaign_prefix','vertical','gender','marital','children_in_home','dwelling_type',
       'homeowner','credit_card_user','senior_in_home','veh1_make','veh1_class']
X = pd.concat([df[NUM].apply(pd.to_numeric, errors='coerce'),
               pd.get_dummies(df[CAT].astype('object').where(df[CAT].notna(),'NA'), prefix=CAT)], axis=1)
print('feature matrix:', X.shape)

## 4 · Model 1 — XGBoost conversion likelihood

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb
y = df['converted'].values
Xtr,Xte,ytr,yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
spw = (ytr==0).sum()/max((ytr==1).sum(),1)
clf = xgb.XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
                        colsample_bytree=0.8, min_child_weight=5, scale_pos_weight=spw,
                        eval_metric='logloss', n_jobs=4, random_state=42)
clf.fit(Xtr,ytr)
p = clf.predict_proba(Xte)[:,1]
print('ROC-AUC: %.4f' % roc_auc_score(yte,p))
print('PR-AUC:  %.4f (base %.3f)' % (average_precision_score(yte,p), yte.mean()))

### Decile lift & feature importance

In [ ]:
d = pd.DataFrame({'y':yte,'p':p}).sort_values('p',ascending=False).reset_index(drop=True)
d['dec'] = (np.arange(len(d))/len(d)*10).astype(int)
lift = d.groupby('dec')['y'].mean()*100
lift.plot(kind='bar', title='Actual conversion %% by predicted decile (0=highest)', ylabel='conversion %')
plt.axhline(100*yte.mean(), ls='--', c='b'); plt.tight_layout(); plt.show()
imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False).head(20)
imp[::-1].plot(kind='barh', figsize=(7,7), title='Top 20 features (gain)'); plt.tight_layout(); plt.show()

## 5 · Model 2 — call value (regression on converted calls)
Value is zero-inflated; model value **given** conversion, combine as expected value = P(convert) x E(value|convert).

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error
m = df['converted']==1
Xv = X[m.values].fillna(0); yv = df.loc[m,'revenue'].values
Xtr2,Xte2,ytr2,yte2 = train_test_split(Xv,yv,test_size=0.25,random_state=42)
reg = Pipeline([('sc',StandardScaler()),('lr',Ridge(alpha=10))]).fit(Xtr2,ytr2)
pv = reg.predict(Xte2)
print('R2: %.4f' % r2_score(yte2,pv))
print('MAE: $%.2f (baseline $%.2f)' % (mean_absolute_error(yte2,pv), mean_absolute_error(yte2, np.full_like(yte2,ytr2.mean()))))
print('Note: near-zero/negative R2 is expected — demographics do not predict $ value; it is set by buyer payout tiers.')

## 7 · Regression coefficient tables (interpretable layer)

Linear/logistic regression on the ICP features. This is the **coefficient score** view:
direction, magnitude, standard error, and p-value for each variable, holding the others constant.

- **Logit** on `converted` → odds-ratio effect of each feature on *likelihood to convert*.
- **OLS** on `revenue` (all calls) → dollar effect of each feature on *revenue per call*.

We standardize numeric features so coefficients are comparable (effect per 1 SD). Missing
enrichment is kept as its own signal via the `enriched` flag + median-fill, so coefficients read
*"among matched callers, holding coverage constant."*

In [ ]:
# statsmodels gives proper coefficient tables (coef, std err, p-value, CI)
!pip -q install statsmodels >/dev/null 2>&1
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# Build a compact, interpretable design matrix (std numeric + key one-hots).
REG_NUM = ['age','income_rank','networth_rank','home_value_usd','education_rank',
           'length_of_residence','investment_interest','vehicle_count','mortgage_ltv','enriched']
REG_CAT = ['vertical','gender','marital','homeowner','credit_card_user']

Xn = df[REG_NUM].apply(pd.to_numeric, errors='coerce')
Xn = Xn.fillna(Xn.median())
Xc = pd.get_dummies(df[REG_CAT].astype('object').where(df[REG_CAT].notna(),'NA'),
                    prefix=REG_CAT, drop_first=True)
scaler = StandardScaler()
Xn_s = pd.DataFrame(scaler.fit_transform(Xn), columns=REG_NUM, index=df.index)
Xreg = pd.concat([Xn_s, Xc.astype(float)], axis=1)
Xreg = sm.add_constant(Xreg)
print('regression design matrix:', Xreg.shape)

### 7a · Logistic regression — likelihood to convert
Odds ratio > 1 = raises conversion odds; < 1 = lowers. Sorted by significance.

In [ ]:
logit = sm.Logit(df['converted'].astype(float), Xreg).fit(disp=0)
coef = pd.DataFrame({
    'coef': logit.params,
    'odds_ratio': np.exp(logit.params),
    'std_err': logit.bse,
    'p_value': logit.pvalues,
})
coef['sig'] = pd.cut(coef['p_value'], [0,0.001,0.01,0.05,1], labels=['***','**','*',''])
coef = coef.drop('const', errors='ignore').sort_values('p_value')
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
print('Logit pseudo-R2: %.4f  |  n=%d' % (logit.prsquared, int(logit.nobs)))
coef.round(4)

### 7b · OLS — dollars of revenue per call
Coefficient = change in `$ revenue/call` per 1-SD of a numeric feature (or vs the baseline level
for categoricals). This is the exec-friendly "worth $X more/less" table.

In [ ]:
ols = sm.OLS(df['revenue'].astype(float), Xreg).fit(cov_type='HC3')  # robust SE
coef2 = pd.DataFrame({'coef_$': ols.params, 'std_err': ols.bse, 'p_value': ols.pvalues})
coef2['sig'] = pd.cut(coef2['p_value'], [0,0.001,0.01,0.05,1], labels=['***','**','*',''])
coef2 = coef2.drop('const', errors='ignore').sort_values('coef_$', ascending=False)
print('OLS R2: %.4f (adj %.4f)  |  n=%d' % (ols.rsquared, ols.rsquared_adj, int(ols.nobs)))
print('Reminder: low R2 is expected — $ value is largely set by buyer payout tiers, not demographics.')
coef2.round(4)

## 8 · SHAP — what the XGBoost model actually keys on (signal-detection layer)

SHAP decomposes each prediction into additive per-feature contributions, so it reads almost like a
coefficient but **captures nonlinear + interaction effects** the regression misses. Uses the
`clf` XGBoost model trained in Section 4.

- **Beeswarm**: global importance + direction (red = high feature value).
- **Bar**: mean |SHAP| ranking.
Compare these to the Section 7 tables: features that agree = robust signal; features that only show
up in SHAP = nonlinear/threshold/interaction effects worth a closer look.

In [ ]:
!pip -q install shap >/dev/null 2>&1
import shap
# Use the held-out test set from Section 4 (clf, Xte must exist)
explainer = shap.TreeExplainer(clf)
sv = explainer.shap_values(Xte)
shap.summary_plot(sv, Xte, max_display=20, show=True)          # beeswarm
shap.summary_plot(sv, Xte, plot_type='bar', max_display=20, show=True)  # mean |SHAP|

### 8a · SHAP vs regression — agreement table
Side-by-side of mean |SHAP| rank (XGBoost) against the logit p-value / odds ratio, so you can
instantly see where the two methods corroborate each other.

In [ ]:
mean_abs = pd.Series(np.abs(sv).mean(0), index=Xte.columns).sort_values(ascending=False)
cmp = pd.DataFrame({'shap_mean_abs': mean_abs})
cmp['shap_rank'] = cmp['shap_mean_abs'].rank(ascending=False).astype(int)
cmp = cmp.join(coef[['odds_ratio','p_value','sig']])  # from logit in 7a
cmp.head(20).round(4)

## 6 · Google Ads demographics (optional)
Upload `google_demographics.csv` to compare Google's observed demographics with the ICP enrichment.

In [ ]:
try:
    gd = pd.read_csv('google_demographics.csv')
    gd['cpa'] = gd['cost']/gd['conversions'].replace(0,np.nan)
    for dim in ['age','income']:
        sub = gd[gd.dimension==dim]
        sub.plot(x='segment', y='cpa', kind='bar', title=f'Google CPA by {dim}', legend=False, ylabel='$ cost/conv')
        plt.tight_layout(); plt.show()
except FileNotFoundError:
    print('upload google_demographics.csv to run this cell')